# Math Foundations

- [Linear Algebra](#Linear-Algebra)
  - [Vectors and basic operations](#Vectors-and-basic-operations)
  - [Matrix multiplication, worked by hand](#Matrix-multiplication,-worked-by-hand)
  - [Transpose and identity](#Transpose-and-identity)
  - [Matrix inverse, worked by hand (2x2)](#Matrix-inverse,-worked-by-hand-(2x2))
  - [Eigenvalues and eigenvectors, brief (fuller worked treatment in `10-unsupervised.ipynb`'s PCA section)](#Eigenvalues-and-eigenvectors,-brief-(fuller-worked-treatment-in-10-unsupervised.ipynb's-PCA-section))
  - [Vector norms, and the direct tie to L1/L2 regularization](#Vector-norms,-and-the-direct-tie-to-L1/L2-regularization)
- [Calculus and Differentiation](#Calculus-and-Differentiation)
  - [Derivatives: rate of change](#Derivatives:-rate-of-change)
  - [Partial derivatives](#Partial-derivatives)
  - [The chain rule, worked exactly as the logreg gradient derivation used it](#The-chain-rule,-worked-exactly-as-the-logreg-gradient-derivation-used-it)
  - [Gradient: the vector of all partial derivatives](#Gradient:-the-vector-of-all-partial-derivatives)
  - [Jacobian and Hessian](#Jacobian-and-Hessian)
  - [Gradient descent and the learning rate](#Gradient-descent-and-the-learning-rate)

### Linear Algebra

The math underneath every `W @ x` in this series. Vectors, matrices, matrix multiplication, inverse, eigenvalues/eigenvectors (fuller treatment in `10-unsupervised.ipynb`'s PCA section, brief version here), norms (tie directly to `05-loss-regularization.ipynb`'s L1/L2 penalties).

#### Vectors and basic operations

A vector is an ordered list of numbers, e.g. a 2-feature document representation x = [0.6, 0.7] (mentions_IRS-strength, urgency-strength, toy values).
```
addition:       [1,2] + [3,4] = [4,6]           (element-wise)
scalar multiply: 2 * [1,2] = [2,4]               (every element scaled)
dot product:    [1,2] . [3,4] = 1*3 + 2*4 = 11   (sum of element-wise products, a single number)
```
The dot product is the single most-used operation in this entire series, it is literally what `w . x` computes in every logistic regression score, every attention score in `05-tokens-arch.ipynb`, every leaf-split decision under the hood.

In [ ]:
import numpy as np

a = np.array([1, 2])
b = np.array([3, 4])

print("addition:", a + b)
print("scalar multiply:", 2 * a)
print("dot product:", np.dot(a, b), "(or a @ b:", a @ b, ")")

#### Matrix multiplication, worked by hand

A matrix is a grid of numbers, shape (rows, columns). Matrix multiplication C = A @ B: entry C[i,j] is the dot product of A's row i and B's column j. Requires A's column count to equal B's row count.

Worked example, same shape logic as `08-classical-ml.ipynb`'s multinomial logreg forward pass: X (2 docs, 3 words) @ W^T (3 words, 2 classes) -> Z (2 docs, 2 classes).
```
X = [[1, 0, 1],      W = [[2, 0, 1],     W^T = [[2, 1],
     [0, 1, 1]]           [1, 1, 0]]            [0, 1],
                                                  [1, 0]]

Z[0,0] = X_row0 . W^T_col0 = [1,0,1].[2,0,1] = 1*2+0*0+1*1 = 3
Z[0,1] = X_row0 . W^T_col1 = [1,0,1].[1,1,0] = 1*1+0*1+1*0 = 1
Z[1,0] = X_row1 . W^T_col0 = [0,1,1].[2,0,1] = 0*2+1*0+1*1 = 1
Z[1,1] = X_row1 . W^T_col1 = [0,1,1].[1,1,0] = 0*1+1*1+1*0 = 1

Z = [[3, 1],
     [1, 1]]
```
Every entry is one dot product. This is literally what `X @ W.T` computes internally, one dot product per (document, class) pair, exactly the shape mechanics used throughout `08-classical-ml.ipynb` and `09-trees.ipynb`.

In [ ]:
X = np.array([[1, 0, 1], [0, 1, 1]])
W = np.array([[2, 0, 1], [1, 1, 0]])

Z = X @ W.T
print("Z:\n", Z)
print("shapes: X", X.shape, "@ W.T", W.T.shape, "->", Z.shape)

#### Transpose and identity

Transpose (A^T): flip rows and columns, A[i,j] becomes A^T[j,i]. Used constantly just to make shapes line up for multiplication (X @ W.T above needed the transpose specifically to make the inner dimensions match).

Identity matrix (I): the matrix equivalent of the number 1, ones on the diagonal, zeros elsewhere, A @ I = A for any compatible A. Used in `05-loss-regularization.ipynb`'s Ridge derivation (the lambda*I term added before inverting) and shows up any time "add a small constant to the diagonal for numerical stability" is needed.

#### Matrix inverse, worked by hand (2x2)

A^-1 such that A @ A^-1 = I. For a 2x2 matrix [[a,b],[c,d]], the formula is direct: A^-1 = (1/det(A)) * [[d,-b],[-c,a]], where det(A) = ad-bc.

Worked example, the matrix from `08-classical-ml.ipynb`'s normal equation setup, A = [[4, 2], [2, 3]]:
```
det(A) = 4*3 - 2*2 = 12-4 = 8
A^-1 = (1/8) * [[3, -2], [-2, 4]] = [[0.375, -0.25], [-0.25, 0.5]]
```
This is exactly the operation `np.linalg.inv(X_design.T @ X_design)` performs inside linear regression's normal equation `w = (X^T X)^-1 X^T y`, solving for the weights directly without any iterative gradient descent. If det(A)=0, the matrix has NO inverse (singular), which is exactly what happens under severe multicollinearity (near-duplicate features), flagged as an assumption violation in `08-classical-ml.ipynb`'s Linear Regression section.

In [ ]:
A = np.array([[4, 2], [2, 3]])
det_A = np.linalg.det(A)
A_inv = np.linalg.inv(A)

print("det(A):", det_A)
print("A inverse:\n", A_inv)
print("check, A @ A_inv (should be identity):\n", np.round(A @ A_inv, 6))

#### Eigenvalues and eigenvectors, brief (fuller worked treatment in `10-unsupervised.ipynb`'s PCA section)

For a square matrix A, an eigenvector v and eigenvalue lambda satisfy A @ v = lambda * v, applying A to v just scales it, does not change its direction. PCA's entire mechanism (finding the directions of maximum variance) is exactly finding the eigenvectors of the covariance matrix, the eigenvector with the largest eigenvalue IS the direction PC1 points in, already worked through with real numbers in `10-unsupervised.ipynb`, not repeated here.

#### Vector norms, and the direct tie to L1/L2 regularization

L2 norm: ||v||_2 = sqrt(sum(v_i^2)), the ordinary (Euclidean) length of a vector. This is EXACTLY the sum(w^2) term in Ridge regression's penalty (see `05-loss-regularization.ipynb`), Ridge's penalty is literally lambda times the squared L2 norm of the weight vector.

L1 norm: ||v||_1 = sum(|v_i|), the sum of absolute values. This is EXACTLY Lasso's penalty term, lambda times the L1 norm of the weight vector.

Worked example, w = [0.3, 0.8] (the same toy weights from `05-loss-regularization.ipynb`'s L1/L2 worked comparison):
```
||w||_2 = sqrt(0.3^2 + 0.8^2) = sqrt(0.09+0.64) = sqrt(0.73) = 0.854
||w||_1 = |0.3| + |0.8| = 1.1
```
The geometric difference between these two norms (L2's circular "equal distance" contour vs. L1's diamond-shaped contour) is the actual geometric reason Lasso can produce exact zeros and Ridge cannot, the diamond shape's corners land exactly on the axes, where one coordinate is exactly zero.

In [ ]:
w = np.array([0.3, 0.8])

print("L2 norm:", np.linalg.norm(w, ord=2))
print("L1 norm:", np.linalg.norm(w, ord=1))

### Calculus and Differentiation

The math underneath every gradient computed by hand across this series, and what `.backward()` in `01-nn-training.ipynb` automates. Derivatives, partial derivatives, the chain rule (the single most important rule here), gradients, Jacobian, Hessian (ties directly to `09-trees.ipynb`'s XGBoost Hessian usage).

#### Derivatives: rate of change

Definition: f'(x) = lim(h->0) [f(x+h) - f(x)] / h, the instantaneous rate of change of f at x, the slope of the tangent line at that exact point.

Worked example, f(x) = x^2, at x=3, approximated with a small but nonzero h=0.001 (this is literally what the limit definition converges to as h shrinks toward 0):
```
f(3) = 9
f(3.001) = 3.001^2 = 9.006001

f'(3) ~ (9.006001 - 9) / 0.001 = 0.006001/0.001 = 6.001
```
The exact analytical derivative of x^2 is 2x, at x=3 that is exactly 6, the tiny-h approximation (6.001) confirms it. This numerical approximation is also how gradient-checking works in practice, verifying an analytically-derived (or autograd-computed) gradient by comparing it against this tiny-step finite-difference estimate.

In [ ]:
def f(x):
    return x ** 2

x, h = 3.0, 0.001
numerical_derivative = (f(x + h) - f(x)) / h
analytical_derivative = 2 * x  # d/dx(x^2) = 2x

print("numerical (finite difference):", numerical_derivative)
print("analytical (2x):", analytical_derivative)

#### Partial derivatives

For a function of MULTIPLE variables, the partial derivative with respect to one variable holds every other variable fixed and measures the rate of change along just that one direction.

Worked example, f(w,x) = w*x + w^2 (a toy loss-like function of a weight w and an input x), at w=2, x=3:
```
df/dw = x + 2w    (treating x as constant)   = 3 + 2*2 = 7
df/dx = w         (treating w as constant)   = 2
```
This is exactly how `dL/dw` is computed throughout this series while `dL/db` is computed separately, each partial derivative asks "if I nudge ONLY this one parameter, holding everything else fixed, how much does the output change."

In [ ]:
import sympy as sp

w, x = sp.symbols("w x")
f_expr = w * x + w**2

df_dw = sp.diff(f_expr, w)
df_dx = sp.diff(f_expr, x)

print("df/dw =", df_dw, "-> at w=2,x=3:", df_dw.subs({w: 2, x: 3}))
print("df/dx =", df_dx, "-> at w=2,x=3:", df_dx.subs({w: 2, x: 3}))

#### The chain rule, worked exactly as the logreg gradient derivation used it

Formula: if y = f(g(x)), then dy/dx = df/dg * dg/dx, differentiate the outer function, then multiply by the derivative of the inner function. This is THE rule that makes training any layered model (logistic regression's loss-through-sigmoid, or a full neural network's loss-through-many-layers) possible at all.

Worked, reproducing `08-classical-ml.ipynb`'s from-scratch logreg derivation step by step: L depends on p, p depends on z, z depends on w. Three nested functions, chain rule links them:
```
dL/dw = dL/dp * dp/dz * dz/dw
```
At x=2, w=0.5, b=0.1, y=1 (the exact setup used in `01-nn-training.ipynb`'s autograd verification):
```
z = 1.1, p = sigmoid(1.1) = 0.7503

dL/dp = (p-y)/(p*(1-p))   [derivative of -[y*ln(p)+(1-y)*ln(1-p)] w.r.t. p]
      = (0.7503-1)/(0.7503*0.2497) = -0.2497/0.1873 = -1.333
dp/dz = p*(1-p) = 0.7503*0.2497 = 0.1873          [sigmoid's own derivative]
dz/dw = x = 2                                       [z=w*x+b, so dz/dw=x]

dL/dw = (-1.333) * (0.1873) * (2) = -0.4994
```
The (p-y)/(p*(1-p)) and p*(1-p) terms cancel exactly, leaving the clean (p-y)*x result derived directly in `08-classical-ml.ipynb`, and confirmed numerically identical there via PyTorch autograd. Three chain-rule steps multiplied together, that IS backpropagation for this one-layer case, a full neural network just chains many more of these steps, one per layer.

In [ ]:
x_val, w_val, b_val, y_val = 2.0, 0.5, 0.1, 1.0

z = w_val * x_val + b_val
p = 1 / (1 + (2.71828 ** -z))

dL_dp = (p - y_val) / (p * (1 - p))
dp_dz = p * (1 - p)
dz_dw = x_val

dL_dw_chain = dL_dp * dp_dz * dz_dw
dL_dw_simplified = (p - y_val) * x_val

print("chain rule, 3 separate terms multiplied:", dL_dw_chain)
print("simplified closed form (p-y)*x:", dL_dw_simplified)
print("match:", round(dL_dw_chain, 4) == round(dL_dw_simplified, 4))

#### Gradient: the vector of all partial derivatives

The gradient (written grad-f or nabla-f) is just every partial derivative stacked into one vector, [df/dw1, df/dw2, ..., df/dwn]. "Gradient descent" (used in every `for i in range(n_iters)` loop across this series) means stepping in the direction OPPOSITE this vector, since the gradient points in the direction of steepest INCREASE, stepping opposite it decreases the loss fastest.

#### Jacobian and Hessian

Jacobian: the matrix of partial derivatives for a VECTOR-valued function (multiple outputs, not just one), each row is the gradient of one output with respect to every input. Multi-class logreg's softmax output is exactly this case, K outputs, the Jacobian describes how every output probability changes with respect to every input logit simultaneously.

Hessian: the matrix of SECOND derivatives, how the gradient itself is changing, curvature information. This is exactly the H in `09-trees.ipynb`'s XGBoost derivation, h=p(1-p), XGBoost uses this second-derivative (curvature) information directly in its leaf-weight and Gain formulas, which is precisely why it converges faster than classic GBM's first-order-only approach (also covered in `09-trees.ipynb`).

In [ ]:
import torch

w = torch.tensor(0.5, requires_grad=True)
x, b, y = torch.tensor(2.0), torch.tensor(0.1), torch.tensor(1.0)

z = w * x + b
p = torch.sigmoid(z)
loss = -(y * torch.log(p) + (1 - y) * torch.log(1 - p))

grad = torch.autograd.grad(loss, w, create_graph=True)[0]
hessian = torch.autograd.grad(grad, w)[0]

print("gradient dL/dw:", grad.item())
print("second derivative (Hessian, 1x1 here) d2L/dw2:", hessian.item())

#### Gradient descent and the learning rate

θ←θ−η∇J. Linear (MSE): ∇=(1/m)Xᵀ(Xθ−y). Logistic: ∇=(1/m)Xᵀ(σ(Xθ)−y) — same form. η too small: slow; too large: oscillates/diverges. Scale features; use mini-batch/Adam; both losses convex (global minimum).

Run it: a small step crawls, a moderate step converges in a few iterations, and a step above 2 divided by the largest curvature (about 0.13 here) overshoots further each time, so the loss explodes.

In [1]:
import numpy as np
X = np.array([1.0, 2.0, 3.0, 4.0]); y = np.array([2.0, 4.0, 6.0, 8.0])   # true w = 2
def run(lr, steps=8):
    w, hist = 0.0, []
    for _ in range(steps):
        grad = np.mean(2 * (w * X - y) * X)                          # d/dw of mean squared error
        w -= lr * grad
        hist.append(round(float(np.mean((w * X - y) ** 2)), 2))
    return hist
for lr in (0.01, 0.1, 0.25):
    print(f"learning rate {lr:<5}: loss by step {run(lr)}")

learning rate 0.01 : loss by step [21.67, 15.66, 11.31, 8.17, 5.91, 4.27, 3.08, 2.23]
learning rate 0.1  : loss by step [7.5, 1.88, 0.47, 0.12, 0.03, 0.01, 0.0, 0.0]
learning rate 0.25 : loss by step [226.88, 1715.74, 12975.3, 98125.71, 742075.67, 5611947.26, 42440351.13, 320955155.4]
